# SDF for quadratic BSpline

A segment of $B^{(2)}$-Spline between $c_{j-1}, c_{j}, c_{j+1}$ is equivalent to $B^{(2)}$-ezier:

- $c_1 = ½(c_{j-1} + c_j)$
- $c_2 = c_j$
- $c_3 = ½(c_{j+1} + c_j)$

$F_{B_1 \dots B_n}: p_s \to \{\tilde{t}, \tilde{p}, d^2, s\}$ for the closest segment in the spline

- $B_1 \dots B_n$ — bezier segments of the bspline
- $p_s$ — sample point in UV
- $\tilde{t}$ — param along curve
- $\tilde{p}$ — closest point on curve, $ = S(\tilde{t})$
- $d^2$ — squared distance, $= |p_s - \tilde{p}|^2$
- $s = ±1$ — side from curve direction


In [ ]:
%%html
<style>
    :root {
        --jp-content-font-color0: var(--vscode-editor-foreground);
        --jp-content-font-color1: var(--vscode-editor-foreground);
        --jp-widgets-color: var(--vscode-editor-foreground);
        --jp-widgets-input-color: var(--vscode-editor-foreground);
        --jp-widgets-input-background-color: var(--vscode-editor-background);
        --jp-widgets-font-size: var(--vscode-editor-font-size);
    }
    .jupyter-widgets input {
        background-color: var(--jp-widgets-input-background-color);
    }
    .cell-output-ipywidget-background {
        background-color: transparent !important;
    }
</style>


In [ ]:
from typing import Iterable, NamedTuple
from numpy.typing import NDArray

import numpy as np
from numpy.random import random as nprand
import ipywidgets as wg
import k3d

from utils import npvec, arr, arrgs, garr, unflat, unflat_np, f32, disquance, normalize

inf = 100500.0

In [ ]:
def pladd(plot, *args):
    for a in args:
        plot.__iadd__(a)


def togglevis(obj, val):
    obj.visible = val

# Piece-wisdom


In [ ]:
def controls2_iter(controls: NDArray):
    """Iterate over all possible sets of controls"""
    for i in range(len(controls) - 2):
        yield controls[i : i + 3]

# Bezier Segment

Nearly flat segment:

- solving in 2d projection
- restoring 3d for bumping

Centering around $c2$ and $t \in [-0.5, +0.5]$

- $c_2 \to 0$
- $c_1 \to v_1 = c_1 - c_2$
- $c_3 \to v_3 = c_3 - c_2$
- $v_a = v_3 + v_1$ — parabolic axis
- $v_d = v_3 - v_1$ — parabolic direction
- $c_a = S(0) - c_2 = .25 v_a$ ­— apex of the parabola (closest to $c_2$, max curvature)
- $p_s = p - c_2$
- $p_c = c_a - p_s$ ­— sample relative to apex

$$
S(t') - c_2 =
\big[1, t', t'^2 \big]
\begin{bmatrix}
c_a \\
v_d \\
v_a \\
\end{bmatrix}
$$

$$
\frac{dS}{dt}(t') =
\big[1, t' \big]
\begin{bmatrix}
v_d \\
2 v_a \\
\end{bmatrix}
$$

### Orientation

- $O$ — vector from projection to sample
- $T$ — tangential vector at projection point
- $[O×T]_z = O_x T_y - O_y T_x$

$$
\begin{cases}
[O×T]_z > 0 \implies \text{O points to right-hand side} \\
[O×T]_z < 0 \implies \text{O points to left-hand side} \\
\tag{orientation}
\end{cases}
$$

### Horizon

$$
\begin{cases}
p_s · v_1 > v_1 · v_1 \implies t'_{prj} < -0.5
\\
p_s · v_3 > v_3 · v_3 \implies t'_{prj} > +0.5
\end{cases}
\tag{visibility}
$$


In [ ]:
from typing import Self


class Bezielt(NamedTuple):
    """Centered at c_2 and t=-0.5..+0.5"""

    c2: npvec
    ca: npvec
    v1: npvec
    v3: npvec
    va: npvec
    vd: npvec

    j: int = 0  # index of segm in spline

    @classmethod
    def BSeg(cls, bcontrols: NDArray, j: int = 0) -> Self:
        assert bcontrols.shape[0] == 3
        p1 = bcontrols[0]
        p2 = bcontrols[1]
        p3 = bcontrols[2]
        c2 = p2
        v1 = (p1 - p2) * 0.5
        v3 = (p3 - p2) * 0.5
        va = v3 + v1
        vd = v3 - v1
        ca = 0.25 * va
        return cls(c2, ca, v1, v3, va, vd, j)

    @classmethod
    def Flat(cls, orig: Self) -> Self:
        return cls(orig.c2[:2], orig.ca[:2], orig.v1[:2], orig.v3[:2], orig.va[:2], orig.vd[:2], orig.j)

    def loc(self, point: npvec | NDArray) -> npvec:
        return point - self.c2

    def glb(self, point: npvec | NDArray) -> npvec:
        return point + self.c2

    def point(self, t: float) -> npvec:
        return t * t * self.va + t * self.vd + self.ca

    def flow(self, t: float) -> npvec:
        return 2 * self.va * t + self.vd

    def curve(self, tspace: NDArray) -> NDArray:
        return garr(self.point(t) for t in tspace)


def b2segments(controls: NDArray) -> tuple[Bezielt, ...]:
    """Enumerated segments with j = 1...N"""
    assert controls.shape[0] > 3
    return tuple(Bezielt.BSeg(ctrl, j) for j, ctrl in enumerate(controls2_iter(controls), start=1))

In [ ]:
def orientation(forw: npvec, targ: npvec) -> int:
    return int(np.sign(targ[1] * forw[0] - targ[0] * forw[1]))

In [ ]:
def horizon(bezier2d: Bezielt, sample: npvec) -> int:
    """number of edges in potential reachability, 0 = OOB"""
    h1 = bezier2d.v1 @ bezier2d.v1
    h3 = bezier2d.v3 @ bezier2d.v3
    p = bezier2d.loc(sample)
    p1 = bezier2d.v1 @ p
    p3 = bezier2d.v3 @ p
    return int(p1 < h1) + int(p3 < h3)

### Projecting

$$
[1, t, t^2, t^3]
\begin{pmatrix}
p_c·v_d \\
2 p_c·v_a + v_d·v_d\\
3 v_a·v_d \\
2 v_a·v_a \\
\end{pmatrix}
=0
\tag{perpendicularity}
$$

Assuming the segment is nearly flat.

Solving in $XY$ 2D coords, restoring $z$ for projection.


In [ ]:
from math import sqrt, cbrt, cos, acos, pi


def solve(bezier: Bezielt, p: npvec) -> tuple[float, ...]:
    va = bezier.va
    vd = bezier.vd
    pc = bezier.ca - p

    aa = float(va @ va)
    ad = float(va @ vd)
    dd = float(vd @ vd)
    pa = float(pc @ va)
    pd = float(pc @ vd)

    a_1 = dd + 2 * pa
    p3 = (2 * aa * a_1 - 3 * ad**2) / (12 * aa**2)
    q2 = (ad**3 - aa * ad * a_1 + 2 * aa**2 * pd) / (8 * aa**3)
    off = ad / (2 * aa)

    D = p3**3 + q2**2

    if D > 0:
        C = cbrt(sqrt(D) - q2)
        t_ = C - p3 / C
        return (t_ - off,)
    else:
        rt = sqrt(-p3)
        k = 2 * rt
        ph = 2 * pi / 3
        th = acos(q2 / (p3 * rt)) / 3
        t_0 = k * cos(th)
        t_1 = k * cos(th - ph)
        t_2 = k * cos(th - 2 * ph)
        return (t_0 - off, t_1 - off, t_2 - off)

In [ ]:
class Projection(NamedTuple):
    segm: Bezielt | None
    t: float
    pnt: npvec
    dsq: float = 0
    sgn: int = 0

    @property
    def sdist(self):
        return self.sgn * np.sqrt(self.dsq)


nullproj = Projection(None, 0, np.zeros(3), 0, 0)

In [ ]:
def project1(bezier3d: Bezielt, sample: npvec) -> Projection:
    bezier2d = Bezielt.Flat(bezier3d)
    ps = bezier2d.loc(sample[:2])
    tt = solve(bezier2d, ps)
    tt = tuple(filter(lambda t: -0.5 <= t <= 0.5, tt))

    if len(tt) == 0:
        return nullproj

    if len(tt) == 1:
        t = tt[0]
        pnt = bezier2d.point(t)
        dsq = disquance(pnt, ps)
    else:
        pnts = tuple(bezier2d.point(t) for t in tt)
        dsqs = tuple(disquance(p, ps) for p in pnts)
        best = np.argmin(dsqs)
        t = tt[best]
        pnt = pnts[best]
        dsq = dsqs[best]

    # if -0.5 > t or t > +0.5:
    #     return nullproj

    sgn = orientation(bezier2d.flow(t), ps - pnt)
    pnt3d = bezier3d.glb(bezier3d.point(t))
    return Projection(bezier3d, t, pnt3d, dsq, sgn)


In [ ]:
def projectall(segments: Iterable[Bezielt], sample: npvec) -> Projection:
    projs = [project1(segm, sample) for segm in segments]
    projs = [prj for prj in projs if prj.segm is not None]

    if len(projs) == 0:
        return nullproj

    dsqs = [prj.dsq for prj in projs]
    best = int(np.argmin(dsqs))
    return projs[best]

In [ ]:
def projectviz(segments: Iterable[Bezielt], sample: npvec) -> Projection:
    """only visible before edge horizon"""
    projs = [project1(segm, sample) for segm in segments if horizon(segm, sample)]
    projs = [prj for prj in projs if prj.segm is not None]

    if len(projs) == 0:
        return nullproj

    dsqs = [prj.dsq for prj in projs]
    best = int(np.argmin(dsqs))
    return projs[best]


---


In [ ]:
H = 0.25


def random_points(n: int):
    l = np.linspace(-0.75, 0.75, n)
    points = arrgs(l, l, np.zeros(n)).T
    points[1:-1] += (nprand((n - 2, 3)) - 0.5) * arrgs(1.0, 1.0, H)
    return f32(points)

In [ ]:
N = 5
controls = random_points(N)

tspace = np.linspace(-0.5, +0.5, 16, dtype=np.float32)
tspace[0] += 0.001
tspace[-1] -= 0.001

# Plot


In [ ]:
plot = k3d.Plot(
    height=720,
    background_color=0x404040,
    grid_color=0x383838,
    label_color=0x000000,
    menu_visibility=False,
    grid=(-1.0, -1.0, 0.0, 1.0, 1.0, 0.5),
    grid_auto_fit=False,
    mode="callback",
)
plot.layout = wg.Layout(width="720px", height="720px")
# plot.camera_auto_fit = False
# plot.camera = [0, 0, 5, 0, 1, 0, 0, 0, 0]

In [ ]:
randomize_btn = wg.Button(description="randomize")
sample_btn = wg.Button(description="sample")


curve_sw = wg.Dropdown(description="curve", options=[None, "2d", "3d"], value="3d")
image_sw = wg.Dropdown(description="image", options=[None, "sdf", "sqf", "t", "z", "j", "hrz", "viz"], value=None)
controls_sw = wg.Checkbox(description="controls", value=False)
tangents_sw = wg.Checkbox(description="tangents", value=False)

output = wg.Output()

In [ ]:
wg.VBox([
    wg.HBox([randomize_btn, sample_btn]),
    wg.HBox([
        plot,
        wg.VBox([
            curve_sw,
            image_sw,
            controls_sw,
            tangents_sw,
        ]),
    ]),
    output,
])

In [ ]:
k3controls = k3d.line(vertices=controls, color=0x808080, line_width=0.125, shader="thick", visible=False)
k3curve = k3d.line(vertices=[], shader="mesh", color=0xF0F0F0, line_width=0.25, color_map=k3d.colormaps.matplotlib_color_maps.Rainbow, color_range=[-0.5, +0.5])
k3flow = k3d.vectors(origins=[(0, 0, 0)], vectors=[(0, 0, 0)], use_head=False, color=0x000000, head_color=0xF0F0F0, visible=False)
k3points = k3d.points(positions=[], shader="mesh", point_size=0.03125, color_map=k3d.colormaps.matplotlib_color_maps.Seismic, color_range=[-1.0, +1.0])
k3proj = k3d.line(vertices=[], attribute=[], shader="thick", line_width=0.125, color_map=k3d.colormaps.matplotlib_color_maps.Seismic, color_range=[-1.0, +1.0])
k3image = None

pladd(plot, k3proj, k3points, k3controls, k3curve, k3flow)

In [ ]:
controls_sw.observe(lambda ch: togglevis(k3controls, ch.new), "value")
tangents_sw.observe(lambda ch: togglevis(k3flow, ch.new), "value")

In [ ]:
def render_curve3d():
    segments = b2segments(controls)
    k3curve.vertices = np.concat([bezier.glb(bezier.curve(tspace)) for bezier in segments])
    k3curve.attribute = np.tile(tspace, len(segments))
    k3flow.origins = garr(bezier.glb(bezier.point(0.0)) for bezier in segments)
    k3flow.vectors = garr(normalize(bezier.flow(0.0)) for bezier in segments)


def render_curve2d():
    segments = [Bezielt.Flat(b) for b in b2segments(controls)]

    k3curve.vertices = np.concat([unflat_np(bezier.glb(bezier.curve(tspace))) for bezier in segments])
    k3curve.attribute = np.tile(tspace, len(segments))
    k3flow.origins = unflat_np(garr(bezier.glb(bezier.point(0.0)) for bezier in segments))
    k3flow.vectors = unflat_np(garr(normalize(bezier.flow(0.0)) for bezier in segments))


def render_curve(mode: str | None):
    if mode is None:
        k3curve.visible = False
        return
    elif mode == "3d":
        render_curve3d()
    elif mode == "2d":
        render_curve2d()
    k3curve.visible = True
    k3flow.visible = tangents_sw.value

In [ ]:
render_curve(curve_sw.value)
curve_sw.observe(lambda ch: render_curve(ch.new), "value")

In [ ]:
def randomize():
    controls[:] = random_points(N)
    k3controls.vertices = f32(controls)
    k3controls.visible = controls_sw.value

    render_curve(curve_sw.value)
    if k3image:
        render_image(image_sw.value)

    k3proj.visible = False
    k3points.visible = False

In [ ]:
randomize_btn.on_click(lambda _: randomize())

In [ ]:
def sample_random():
    segments = b2segments(controls)
    sample = unflat(nprand(2) - 0.5)
    proj = projectall(segments, sample)

    if curve_sw.value == "2d":
        pnt = unflat(proj.pnt[:2])
    else:
        pnt = proj.pnt

    k3proj.vertices = [sample, pnt]
    k3proj.attribute = [proj.sdist, proj.sdist]
    k3points.positions = [sample, pnt]
    k3points.attribute = [proj.sdist, proj.sdist]

In [ ]:
sample_btn.on_click(lambda btn: sample_random())

## Texture


In [ ]:
RES = 64
PIX = 1.0 / RES
X, Y = np.meshgrid(np.arange(-0.5, 0.5, PIX), np.arange(-0.5, 0.5, PIX))
COORDS = np.stack((Y, X)).T.reshape((RES * RES, 2)) + 0.5 * PIX  # pixel centers

In [ ]:
k3image = k3d.texture(
    attribute=np.random.random((RES, RES)), interpolation=False, color_map=k3d.colormaps.matplotlib_color_maps.Seismic, color_range=[-1.0, 1.0]
)

pladd(plot, k3image)
plot.colorbar_object_id = k3image.id

In [ ]:
def render_sdf(projmap: Iterable[Projection]):
    k3image.color_map = k3d.matplotlib_color_maps.Seismic
    k3image.color_range = [-1.0, +1.0]
    k3image.attribute = garr(prj.sdist for prj in projmap).reshape((RES, RES))


def render_sqf(projmap: Iterable[Projection]):
    k3image.color_map = k3d.matplotlib_color_maps.Binary
    k3image.color_range = [0.0, 1.0]
    k3image.attribute = garr(prj.dsq for prj in projmap).reshape((RES, RES))


def render_t(projmap: Iterable[Projection]):
    k3image.color_map = k3d.basic_color_maps.Rainbow
    k3image.color_range = [-0.5, +0.5]
    k3image.attribute = garr(prj.t for prj in projmap).reshape((RES, RES))


def render_j(projmap: Iterable[Projection]):
    k3image.color_map = k3d.matplotlib_color_maps.Tab10
    k3image.color_range = [0, 9]
    k3image.attribute = garr(int(prj.segm.j) if prj.segm else 0 for prj in projmap).reshape((RES, RES))


def render_z(projmap: Iterable[Projection]):
    k3image.color_map = k3d.matplotlib_color_maps.Binary_r
    k3image.color_range = [-0.5, 0.5]
    k3image.attribute = garr(prj.pnt[2] / H for prj in projmap).reshape((RES, RES))


COORDS3D = unflat_np(COORDS)


def render_image(mode: str | None):
    k3image.visible = False
    k3image.attribute = []

    if mode is None:
        return

    segments = b2segments(controls)
    projmap = [projectviz(segments, sample) for sample in COORDS3D]

    if mode == "sdf":
        render_sdf(projmap)
    elif mode == "sqf":
        render_sqf(projmap)
    elif mode == "t":
        render_t(projmap)
    elif mode == "j":
        render_j(projmap)
    elif mode == "z":
        render_z(projmap)

    k3image.visible = True

In [ ]:
render_image(image_sw.value)
image_sw.observe(lambda ch: render_image(ch.new), "value")